# $(SASA) Models - Kmeans$

In [2]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pickle
import numpy as np
import pandas as pd

from pmbrl.model2 import Base_Line_Simple_Model
# from pmbrl.model2 import Model, Regularized_Reference_Loss
from pmbrl.data import Experiment_Data, get_data_expanded

In [3]:
nome_do_arquivo = 'kmodels.pkl'

with open(nome_do_arquivo, 'rb') as arquivo:
    exp = pickle.load(arquivo)
    data = exp['data']
    models = exp['model']

del exp
del arquivo

In [4]:
data = Experiment_Data()

data.load(path='../testing_data.csv')

expansions = {
    's': ['s0', 's1', 's2', 's3'],
    's_': ['s_0', 's_1', 's_2', 's_3'],
    's__': ['s__0', 's__1', 's__2', 's__3']
}

df = get_data_expanded(data.build_training_dataset(), expansions)
# df = df.loc[df['episode']<15].copy()
df.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,...,s2,s3,s_0,s_1,s_2,s_3,s__0,s__1,s__2,s__3
1844,10,91,"(0.1501612193777863, 0.2709280492756518)","(-0.019, -0.386, 0.062, 1.369)",1,1.0,"(-0.026, -0.18, 0.089, 0.846)",0.0,1.0,"(-0.03, -0.389, 0.106, 1.458)",...,0.062,1.369,-0.026,-0.180,0.089,0.846,-0.030,-0.389,0.106,1.458
346,42,20,"(0.5511096795536096, 0.8938255112494718)","(0.014, 0.013, -0.172, -0.381)",0,1.0,"(0.014, -0.177, -0.179, -0.189)",1.0,1.0,"(0.01, 0.017, -0.183, -0.471)",...,-0.172,-0.381,0.014,-0.177,-0.179,-0.189,0.010,0.017,-0.183,-0.471
2087,58,98,"(0.1955406148449595, 0.6245501454716056)","(-0.354, -0.326, 0.076, -0.079)",0,1.0,"(-0.361, -0.52, 0.074, 0.193)",0.0,1.0,"(-0.371, -0.714, 0.078, 0.464)",...,0.076,-0.079,-0.361,-0.520,0.074,0.193,-0.371,-0.714,0.078,0.464
2024,30,97,"(0.1590895790518159, 0.4779544798666842)","(0.018, -0.78, 0.002, 1.066)",0,1.0,"(0.003, -0.977, 0.024, 1.387)",0.0,1.0,"(-0.017, -1.173, 0.052, 1.715)",...,0.002,1.066,0.003,-0.977,0.024,1.387,-0.017,-1.173,0.052,1.715
1393,8,70,"(0.2613483729966366, 0.7909281747570069)","(0.045, 1.172, -0.071, -1.252)",0,1.0,"(0.069, 0.982, -0.096, -1.059)",1.0,1.0,"(0.088, 1.174, -0.117, -1.289)",...,-0.071,-1.252,0.069,0.982,-0.096,-1.059,0.088,1.174,-0.117,-1.289


# Predict 

In [5]:
def evaluate(models, df):
    def predict(model):
        prediction_dataset = df.copy()
        prediction_dataset[model.grouped_targets_lables] = prediction_dataset.apply(lambda row: data._predict_from_row(row, model), axis=1, result_type='expand')
        return prediction_dataset

    predictions = [predict(m) for m in models]

    prediction_dataset = df.copy()
    for i, pred in enumerate(predictions):
        prediction_dataset[f'estimated_s_model_{i}'] = pred['estimated_s']
        
        results = data.get_evaluation_metrics(pred, p=False)
        prediction_dataset[f'rse_model_{i}'] = results['rse']
        prediction_dataset[f'rse_normalized_model_{i}'] = results['rse_normalized']

        prediction_dataset[f'rse_s0_model_{i}'] = results['rse_s0']
        prediction_dataset[f'rse_s1_model_{i}'] = results['rse_s1']
        prediction_dataset[f'rse_s2_model_{i}'] = results['rse_s2']
        prediction_dataset[f'rse_s3_model_{i}'] = results['rse_s3']

        prediction_dataset[f'rse_s0_normalized_model_{i}'] = results['rse_s0_normalized']
        prediction_dataset[f'rse_s1_normalized_model_{i}'] = results['rse_s1_normalized']
        prediction_dataset[f'rse_s2_normalized_model_{i}'] = results['rse_s2_normalized']
        prediction_dataset[f'rse_s3_normalized_model_{i}'] = results['rse_s3_normalized']


    return prediction_dataset

In [6]:
def reagroup(prediction_dataset):
    # agg_results = prediction_dataset[['episode'] + [
    #     f'rse_model_{i}' for i,_ in enumerate(models)
    # ]].groupby('episode').mean().reset_index()
    agg_results = prediction_dataset[['episode', 'step'] + [
        f'rse_model_{i}' for i,_ in enumerate(models)
    ]].copy()
    
    agg_results['best_model'] = agg_results.apply(lambda row: np.argmin(row[2:].values), axis=1)
    print(agg_results['best_model'].value_counts())

    prediction_dataset['best_model'] = prediction_dataset.apply(
        lambda row: agg_results.loc[(agg_results['episode']==row['episode']) & (agg_results['step']==row['step'])].best_model.values[0],
        axis=1
    )
    prediction_dataset['best_rse'] = prediction_dataset.apply(lambda row: row[f'rse_model_{row.best_model}'],axis=1)

    return prediction_dataset

In [7]:
def predicts(models, df):
    pre_df = df.copy()
    pre_df[['s_0', 's_1', 's_2', 's_3', 'a_', 's__0', 's__1', 's__2', 's__3']] = pre_df[['s0', 's1', 's2', 's3', 'a', 's_0', 's_1', 's_2', 's_3']]

    prediction_dataset = evaluate(models, pre_df)
    prediction_dataset = reagroup(prediction_dataset)
    final_predictions = evaluate(models, df)
    final_predictions['group'] = prediction_dataset['best_model']

    cols = [
        'estimated_s', 'rse', 'rse_normalized',
        'rse_s0', 'rse_s1', 'rse_s2', 'rse_s3', 
        'rse_s0_normalized', 'rse_s1_normalized',
        'rse_s2_normalized', 'rse_s3_normalized'
    ]

    for c in cols:
        final_predictions[c] = final_predictions.apply(lambda x: x[f'{c}_model_{x.group}'], axis=1)

    return final_predictions[cols]

In [8]:
prediction_dataset = predicts(models, df)
prediction_dataset.head()

best_model
1    931
3    520
2    296
4    205
0    175
Name: count, dtype: int64


,estimated_s,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_s0_normalized,rse_s1_normalized,rse_s2_normalized,rse_s3_normalized
1844,"(-0.031, -0.379, 0.109, 1.251)",0.221,0.509071,0.001,0.010,0.003,0.207,0.546990,0.528515,0.508393,0.452383
346,"(0.013, 0.015, -0.184, -0.337)",0.140,0.506346,0.003,0.002,0.001,0.134,0.549102,0.526549,0.503597,0.446137
2087,"(-0.365, -0.709, 0.076, 0.411)",0.066,0.506189,0.006,0.005,0.002,0.053,0.552270,0.527286,0.505995,0.439206
2024,"(-0.013, -1.172, 0.052, 1.798)",0.088,0.504858,0.004,0.001,0.000,0.083,0.550158,0.526303,0.501199,0.441773
1393,"(0.089, 1.174, -0.117, -1.267)",0.023,0.502700,0.001,0.000,0.000,0.022,0.546990,0.526057,0.501199,0.436553


In [9]:
prediction_dataset.describe()

,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_s0_normalized,rse_s1_normalized,rse_s2_normalized,rse_s3_normalized
count,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000
mean,0.226997,0.511999,0.004537,0.012379,0.006194,0.203887,0.550726,0.529100,0.516053,0.452117
std,0.317798,0.017426,0.009261,0.017427,0.018355,0.292164,0.009779,0.004284,0.044017,0.024999
min,0.002000,0.502008,0.000000,0.000000,0.000000,0.000000,0.545935,0.526057,0.501199,0.434671
25%,0.064000,0.504670,0.001000,0.003000,0.000000,0.052000,0.546990,0.526794,0.501199,0.439120
50%,0.144000,0.506814,0.001000,0.007000,0.001000,0.128000,0.546990,0.527778,0.503597,0.445623
75%,0.258500,0.510570,0.004000,0.013000,0.003000,0.239500,0.550158,0.529253,0.508393,0.455164
max,4.170000,0.707944,0.089000,0.189000,0.261000,3.832000,0.639916,0.572517,1.127098,0.762557


In [10]:
prediction_dataset.rse.mean()

np.float64(0.22699717912552891)